In [14]:
import cv2
import numpy as np

# ======================
# 1. 초기 설정
# ======================
sift = cv2.SIFT_create()

FLANN_INDEX_KDTREE = 1
index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
search_params = dict(checks=50)

flann = cv2.FlannBasedMatcher(index_params, search_params)

MIN_MATCH_COUNT = 10

# ======================
# 2. 기준 이미지
# ======================
img1 = cv2.imread("momopop2.jpg")

if img1 is None:
    raise FileNotFoundError("reference.jpg 경로 확인")

# 🔥 왼쪽 이미지 크기 절반으로 줄이기
img1 = cv2.resize(img1, None, fx=0.5, fy=0.5)

gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
kp1, des1 = sift.detectAndCompute(gray1, None)

# ======================
# 3. 카메라
# ======================
cap = cv2.VideoCapture(0)

while True:

    ret, frame = cap.read()
    if not ret:
        break

    gray2 = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    kp2, des2 = sift.detectAndCompute(gray2, None)

    good = []

    if des2 is not None and len(kp2) > 10:

        matches = flann.knnMatch(des1, des2, k=2)

        # Lowe ratio test
        for m, n in matches:
            if m.distance < 0.75 * n.distance:
                good.append(m)

    matchesMask = None

    # ======================
    # 4. Homography (박스)
    # ======================
    if len(good) >= MIN_MATCH_COUNT:

        src_pts = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1,1,2)
        dst_pts = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1,1,2)

        M, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)

        if M is not None:

            matchesMask = mask.ravel().tolist()

            h, w = gray1.shape[:2]

            pts = np.float32([
                [0,0],
                [0,h-1],
                [w-1,h-1],
                [w-1,0]
            ]).reshape(-1,1,2)

            dst = cv2.perspectiveTransform(pts, M)

    # ======================
    # 5. ⭐ 핵심: 매칭 라인 시각화 (네가 원한 부분)
    # ======================
    result = cv2.drawMatches(
        img1, kp1,
        frame, kp2,
        good,
        None,
        matchColor=(255, 0, 0),   # 선 색 (파란/보라 느낌)
        singlePointColor=None,
        matchesMask=matchesMask,
        flags=2
    )

    cv2.imshow("Matching result", result)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()